# Exit Strategy Comparison

Testing different exit strategies WITHOUT the MVRV trail trigger.

**Options:**
1. Current: MVRV > 2.0 activates 25% trail + 20% stop loss
2. Simple trailing stop: Always-on 25% trail from peak
3. Stop loss only: Just -20% stop loss
4. Take profit + stop loss: +50% TP, -20% SL
5. MVRV hard exit: Sell when MVRV > 2.5 (no trail, just exit)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("Exit Strategy Comparison 🔬")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

df = df[df.index >= '2018-12-15'].dropna()
print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Entry signal (same for all strategies)
RL_Z_THRESHOLD = 0.5

entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > RL_Z_THRESHOLD)
)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

In [ ]:
# Exit strategies

@njit
def exit_mvrv_trail(price_arr, mvrv_arr, entry_idx, mvrv_trigger=2.0, trail_pct=0.25, stop_loss_pct=0.20):
    """Current strategy: MVRV triggers trail"""
    entry_price = price_arr[entry_idx]
    peak_price = entry_price
    trailing_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        current_mvrv = mvrv_arr[j]
        
        if current_price > peak_price:
            peak_price = current_price
        
        pnl = (current_price - entry_price) / entry_price
        
        if not trailing_active and current_mvrv >= mvrv_trigger:
            trailing_active = True
        
        if trailing_active:
            trail_stop = peak_price * (1 - trail_pct)
            if current_price <= trail_stop:
                return j, trail_stop, 'mvrv_trail'
        
        if not trailing_active and pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end_of_data'

@njit
def exit_simple_trail(price_arr, mvrv_arr, entry_idx, trail_pct=0.25):
    """Simple trailing stop - always active from entry"""
    entry_price = price_arr[entry_idx]
    peak_price = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        
        if current_price > peak_price:
            peak_price = current_price
        
        trail_stop = peak_price * (1 - trail_pct)
        if current_price <= trail_stop:
            return j, trail_stop, 'trail'
    
    return len(price_arr) - 1, price_arr[-1], 'end_of_data'

@njit
def exit_stop_loss_only(price_arr, mvrv_arr, entry_idx, stop_loss_pct=0.20):
    """Just stop loss, no other exit"""
    entry_price = price_arr[entry_idx]
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        pnl = (current_price - entry_price) / entry_price
        
        if pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end_of_data'

@njit
def exit_tp_sl(price_arr, mvrv_arr, entry_idx, take_profit_pct=0.50, stop_loss_pct=0.20):
    """Take profit + stop loss"""
    entry_price = price_arr[entry_idx]
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        pnl = (current_price - entry_price) / entry_price
        
        if pnl >= take_profit_pct:
            return j, entry_price * (1 + take_profit_pct), 'take_profit'
        
        if pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end_of_data'

@njit
def exit_mvrv_hard(price_arr, mvrv_arr, entry_idx, mvrv_exit=2.5, stop_loss_pct=0.20):
    """Hard exit when MVRV > threshold (no trail)"""
    entry_price = price_arr[entry_idx]
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        current_mvrv = mvrv_arr[j]
        pnl = (current_price - entry_price) / entry_price
        
        if current_mvrv >= mvrv_exit:
            return j, current_price, 'mvrv_exit'
        
        if pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end_of_data'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, fees=0.001, **kwargs):
    """Generic backtest runner"""
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        
        exit_idx, exit_price, exit_reason = exit_func(price_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'net_return': net_return,
            'days_held': exit_idx - entry_idx,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, trade in trades_df.iterrows():
            equity.append(equity[-1] * (1 + trade['net_return']))
        trades_df['equity_after'] = equity[1:]
    
    return trades_df

def calc_metrics(trades, initial_capital, df):
    """Calculate performance metrics"""
    if len(trades) == 0:
        return {'total_return': 0, 'cagr': 0, 'win_rate': 0, 'sharpe': 0, 'max_dd': 0, 'profit_factor': 0}
    
    total_return = (trades['equity_after'].iloc[-1] / initial_capital) - 1
    
    start_date = trades['entry_date'].iloc[0]
    end_date = trades['exit_date'].iloc[-1]
    years = (end_date - start_date).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    win_rate = (trades['net_return'] > 0).mean()
    
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(4) if len(returns) > 1 and returns.std() > 0 else 0
    
    equity_curve = [initial_capital] + list(trades['equity_after'])
    peak = equity_curve[0]
    max_dd = 0
    for eq in equity_curve:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    winners = trades[trades['net_return'] > 0]['net_return'].sum()
    losers = abs(trades[trades['net_return'] <= 0]['net_return'].sum())
    profit_factor = winners / losers if losers > 0 else float('inf')
    
    # Buy & hold
    bh_return = (df['price'].iloc[-1] / df.loc[trades['entry_date'].iloc[0], 'price']) - 1
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'win_rate': win_rate,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'profit_factor': profit_factor,
        'bh_return': bh_return,
        'n_trades': len(trades),
        'final_equity': trades['equity_after'].iloc[-1]
    }

In [ ]:
# Run all strategies
INITIAL_CAPITAL = 100000

strategies = [
    ('1. MVRV Trail (current)', exit_mvrv_trail, {'mvrv_trigger': 2.0, 'trail_pct': 0.25, 'stop_loss_pct': 0.20}),
    ('2. Simple Trail 25%', exit_simple_trail, {'trail_pct': 0.25}),
    ('3. Simple Trail 20%', exit_simple_trail, {'trail_pct': 0.20}),
    ('4. Simple Trail 30%', exit_simple_trail, {'trail_pct': 0.30}),
    ('5. Stop Loss Only 20%', exit_stop_loss_only, {'stop_loss_pct': 0.20}),
    ('6. TP 50% / SL 20%', exit_tp_sl, {'take_profit_pct': 0.50, 'stop_loss_pct': 0.20}),
    ('7. TP 100% / SL 20%', exit_tp_sl, {'take_profit_pct': 1.00, 'stop_loss_pct': 0.20}),
    ('8. MVRV Hard Exit 2.5', exit_mvrv_hard, {'mvrv_exit': 2.5, 'stop_loss_pct': 0.20}),
    ('9. MVRV Hard Exit 3.0', exit_mvrv_hard, {'mvrv_exit': 3.0, 'stop_loss_pct': 0.20}),
]

results = []

print("STRATEGY COMPARISON")
print("="*120)
print(f"{'Strategy':<25} {'Return':>12} {'CAGR':>10} {'Win Rate':>10} {'Sharpe':>10} {'Max DD':>10} {'PF':>8} {'Trades':>8} {'Final $':>12}")
print("-"*120)

for name, exit_func, kwargs in strategies:
    trades = run_backtest(df, entries, exit_func, INITIAL_CAPITAL, **kwargs)
    metrics = calc_metrics(trades, INITIAL_CAPITAL, df)
    
    print(f"{name:<25} {metrics['total_return']*100:>+11.0f}% {metrics['cagr']*100:>+9.1f}% "
          f"{metrics['win_rate']*100:>9.0f}% {metrics['sharpe']:>10.2f} {metrics['max_dd']*100:>9.0f}% "
          f"{metrics['profit_factor']:>8.2f} {metrics['n_trades']:>8} ${metrics['final_equity']:>11,.0f}")
    
    results.append({'name': name, 'trades': trades, 'metrics': metrics})

print("-"*120)
print(f"{'Buy & Hold':<25} {metrics['bh_return']*100:>+11.0f}%")

In [ ]:
# Find the best strategy
best = max(results, key=lambda x: x['metrics']['total_return'])

print("\n" + "="*60)
print("BEST STRATEGY")
print("="*60)
print(f"\n🏆 {best['name']}")
print(f"   Total Return: {best['metrics']['total_return']*100:+.0f}%")
print(f"   CAGR: {best['metrics']['cagr']*100:+.1f}%")
print(f"   Win Rate: {best['metrics']['win_rate']*100:.0f}%")
print(f"   Sharpe: {best['metrics']['sharpe']:.2f}")
print(f"   Max Drawdown: {best['metrics']['max_dd']*100:.0f}%")
print(f"   $100K → ${best['metrics']['final_equity']:,.0f}")

In [ ]:
# Show trades for simple trail (most likely winner)
simple_trail_result = next(r for r in results if 'Simple Trail 25%' in r['name'])
trades = simple_trail_result['trades']

print("\n" + "="*100)
print("SIMPLE TRAIL 25% - TRADE LOG")
print("="*100)
print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Exit $':>10} {'Return':>10} {'Days':>6} {'Exit Reason':<15}")
print("-"*100)

for _, t in trades.iterrows():
    print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
          f"{t['entry_price']:>10,.0f} {t['exit_price']:>10,.0f} "
          f"{t['net_return']*100:>+9.0f}% {t['days_held']:>6} {t['exit_reason']:<15}")

In [ ]:
# Compare MVRV trail vs Simple trail for the problem trade
print("\n" + "="*80)
print("PROBLEM TRADE COMPARISON (2025-01-09 entry)")
print("="*80)

mvrv_result = next(r for r in results if 'MVRV Trail' in r['name'])
simple_result = next(r for r in results if 'Simple Trail 25%' in r['name'])

# Find the Jan 2025 trade in each
for name, result in [('MVRV Trail', mvrv_result), ('Simple Trail', simple_result)]:
    trades = result['trades']
    jan_trade = trades[trades['entry_date'] >= '2025-01-01']
    if len(jan_trade) > 0:
        t = jan_trade.iloc[0]
        print(f"\n{name}:")
        print(f"  Entry: {t['entry_date'].date()} at ${t['entry_price']:,.0f}")
        print(f"  Exit: {t['exit_date'].date()} at ${t['exit_price']:,.0f}")
        print(f"  Return: {t['net_return']*100:+.1f}%")
        print(f"  Exit reason: {t['exit_reason']}")

In [ ]:
# Visualization
import plotly.graph_objects as go

fig = go.Figure()

# Bar chart of returns
names = [r['name'] for r in results]
returns = [r['metrics']['total_return'] * 100 for r in results]

colors = ['green' if r > results[0]['metrics']['total_return']*100 else 'gray' for r in returns]
colors[0] = 'blue'  # Current strategy

fig.add_trace(go.Bar(
    x=names,
    y=returns,
    marker_color=colors,
    text=[f"{r:+.0f}%" for r in returns],
    textposition='outside'
))

fig.add_hline(y=results[0]['metrics']['total_return']*100, line_dash='dash', line_color='blue',
              annotation_text=f"Current: {results[0]['metrics']['total_return']*100:+.0f}%")

fig.update_layout(
    title='Exit Strategy Comparison - Total Return',
    yaxis_title='Return %',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

current = results[0]['metrics']
best_metrics = best['metrics']

print(f"\nCurrent (MVRV Trail): {current['total_return']*100:+.0f}%")
print(f"Best ({best['name']}): {best_metrics['total_return']*100:+.0f}%")
print(f"Improvement: {(best_metrics['total_return'] - current['total_return'])*100:+.0f}%")

if best_metrics['total_return'] > current['total_return']:
    print(f"\n✅ SIMPLER IS BETTER!")
    print(f"   Recommend switching to: {best['name']}")
else:
    print(f"\n⚠️ Current strategy is still best")